# 37. Evaluation with GroUSE-style Judges

Grade grounded answers on completeness, usefulness and faithfulness, and check that the *judge* behaves.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

LLM judges are only useful if the judge itself is reliable. **GroUSE** (Grounded QA Evaluation) is a benchmark idea: give the judge answers with *known* defects and check that it penalises exactly the right criterion. This notebook builds a small GroUSE-style judge for answer relevancy, completeness, usefulness and faithfulness and **unit-tests** it.

## How it works

1. Define rubrics for each criterion (1-5)
2. Prompt the judge once per criterion
3. Create answer variants with known flaws (incomplete, hallucinated, off-topic)
4. Check each flaw lowers the right score
5. Only trust the judge where it passes

```text
 (question, context, answer variant) --> judge --> {relevancy, completeness, usefulness, faithfulness}
 expectation: hallucinated -> low faithfulness ; incomplete -> low completeness ; off_topic -> low relevancy
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 11:17:17.477000 10407 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


GroUSE (Grounded QA Evaluation) scores answers on several criteria and, importantly, **unit-tests the judge** with deliberately flawed answers. Here: answer relevancy, completeness, usefulness and faithfulness on a 1-5 scale.

In [2]:
CRITERIA = {
    "answer_relevancy": "Does the answer address the question without off-topic content? (5 = fully on topic)",
    "completeness": "Does the answer include all the relevant information from the context? (5 = nothing missing)",
    "usefulness": "Is the answer helpful and clear for a user? (5 = very helpful)",
    "faithfulness": "Is every claim supported by the context, with nothing invented? (5 = fully supported)",
}

def judge(question, contexts, answer_text):
    scores = {}
    for name, rubric in CRITERIA.items():
        out = llm(
            f"You are a strict grader.\nCriterion: {rubric}\n\nContext:\n" + "\n".join(contexts) +
            f"\n\nQuestion: {question}\nAnswer: {answer_text}\n\nReply with a single integer from 1 to 5."
        )
        m = re.search(r"[1-5]", out)
        scores[name] = int(m.group()) if m else None
    return scores

## Judge unit tests
A trustworthy judge must punish known-bad answers. We build a good answer and three degraded versions.

In [3]:
q = "How many dimensions does all-MiniLM-L6-v2 produce?"
ctx = [h["text"] for h in store.search(q, 2)]
variants = {
    "good":        "The all-MiniLM-L6-v2 model produces 384-dimensional vectors.",
    "incomplete":  "It produces vectors.",
    "hallucinated": "It produces 1536-dimensional vectors and was trained by OpenAI on code.",
    "off_topic":   "Paris is the capital of France.",
}
import pandas as pd
pd.DataFrame({name: judge(q, ctx, a) for name, a in variants.items()}).T

,answer_relevancy,completeness,usefulness,faithfulness
good,4,2,4,2
incomplete,3,2,4,2
hallucinated,3,3,4,3
off_topic,2,5,1,2


Expect `good` to score highest everywhere, `hallucinated` to fail **faithfulness**, `incomplete` to fail **completeness** and `off_topic` to fail **relevancy**. If your judge cannot tell these apart, switch to a stronger judge model.

## Turn the expectations into automatic tests

In [4]:
results = {name: judge(q, ctx, a) for name, a in variants.items()}
expectations = [
    ("hallucinated has lower faithfulness than good", results["hallucinated"]["faithfulness"] < results["good"]["faithfulness"]),
    ("incomplete has lower completeness than good", results["incomplete"]["completeness"] < results["good"]["completeness"]),
    ("off_topic has lower relevancy than good", results["off_topic"]["answer_relevancy"] < results["good"]["answer_relevancy"]),
]
for name, passed in expectations:
    print("PASS" if passed else "FAIL", "-", name)
print(f"\njudge passes {sum(p for _, p in expectations)}/{len(expectations)} calibration tests")

FAIL - hallucinated has lower faithfulness than good
FAIL - incomplete has lower completeness than good
PASS - off_topic has lower relevancy than good

judge passes 1/3 calibration tests


## Judge consistency
Run the same item at temperature 0.7 five times; a good judge is stable.

In [5]:
import numpy as np
runs = []
for _ in range(5):
    out = llm(f"Rate faithfulness 1-5 of this answer to the context. Reply with a number only.\nContext: {ctx[0]}\nAnswer: {variants['hallucinated']}", temperature=0.7, use_cache=False)
    m = re.search(r"[1-5]", out); runs.append(int(m.group()) if m else None)
print("scores:", runs, "| std:", np.std([r for r in runs if r]))

scores: [3, 3, 3, 3, 3] | std: 0.0


## Score the real pipeline
Now apply the judge to genuine pipeline outputs.

In [6]:
import pandas as pd
rows = []
for item in load_eval_questions()[:5]:
    response, hits = rag(item["question"], store, k=3)
    rows.append({"question": item["question"][:45], **judge(item["question"], [h["text"] for h in hits], response)})
rep = pd.DataFrame(rows).set_index("question"); display(rep); print(rep.mean().round(2))

,answer_relevancy,completeness,usefulness,faithfulness
question,,,,
How many dimensions does the all-MiniLM-L6-v2,3,4,4,4
What constant is used in the Reciprocal Rank,4,3,4,4
Which reranker model is described as small an,4,4,4,4
What does HyDE stand for and how does it work,4,4,4,4
What is mean reciprocal rank?,4,4,4,4


answer_relevancy    3.8
completeness        3.8
usefulness          4.0
faithfulness        4.0
dtype: float64


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Tests the judge before trusting it | Small judges may fail calibration |
| Criterion-level diagnostics | Rubrics need iteration |
| Cheap to extend with new flaws | Five LLM calls per answer |

## When to use it

Before relying on any LLM-as-judge metric for decisions.

## Experiments to try

- Add negative-rejection tests (answers that should say 'I don't know').
- Use a bigger judge model and compare calibration pass rates.
- Add few-shot examples to the rubric prompts.

## Key takeaways

- Calibrate the judge with known flaws.
- Score criteria separately.
- Distrust judges that fail their own tests.

---
**Next:** [38. End-to-End RAG Evaluation](../38_end-2-end_rag_evaluation/end-2-end_rag_evaluation.ipynb)